<a href="https://colab.research.google.com/github/shetty-23/Computer-vision---Assignment-1/blob/feature%2Ftask-2-deblurring/Assignment_1_Computer_vision.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
import os
import shutil
from pathlib import Path
from sklearn.model_selection import train_test_split


In [24]:
# 1. Setup Paths
base_path = Path("/content/drive/MyDrive/gopro_deblur")
# Pointing to the actual subdirectories containing the .png files
blur_dir = base_path / "blur" / "images"
sharp_dir = base_path / "sharp" / "images"

In [25]:
# Searching for .png files in the nested 'images' directory
filenames = sorted([f.name for f in blur_dir.glob("*.png")])

if len(filenames) == 0:
    print(f"No .png files found in {blur_dir}. Please check your path.")
else:
    print(f"Found {len(filenames)} files. Splitting into train, val, and test...")
    train_files, temp_files = train_test_split(filenames, test_size=0.3, random_state=42)
    val_files, test_files = train_test_split(temp_files, test_size=0.5, random_state=42)
    print(f"Splits created: Train({len(train_files)}), Val({len(val_files)}), Test({len(test_files)})")

def move_to_split(file_list, split_name):
    for fname in file_list:
        for folder in ["blur", "sharp"]:
            dest = base_path / "processed" / split_name / folder
            dest.mkdir(parents=True, exist_ok=True)

            # Source files are in the 'images' subfolder
            src_file = base_path / folder / "images" / fname
            if src_file.exists():
                shutil.copy(src_file, dest / fname)
            else:
                print(f"Warning: Source file not found: {src_file}")

Found 1029 files. Splitting into train, val, and test...
Splits created: Train(720), Val(154), Test(155)


In [29]:
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm

def get_blur_score(image_path):
    image = cv2.imread(str(image_path))
    if image is None: return 0
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    return cv2.Laplacian(gray, cv2.CV_64F).var()

print("Calculating blur scores...")
scores = []
for fname in tqdm(filenames):
    path = blur_dir / fname
    scores.append(get_blur_score(path))

# Create DataFrame for analysis
df_blur = pd.DataFrame({'filename': filenames, 'score': scores})

# Define thresholds for categories (Quantile-based for even distribution)
# You can adjust these fixed values if you prefer specific variance ranges
low, high = df_blur['score'].quantile([0.33, 0.66])

def categorize(s):
    if s <= low: return 'Severe'
    if s <= high: return 'Medium'
    return 'Mild'

df_blur['category'] = df_blur['score'].apply(categorize)
print("\nBlur Category Distribution:")
print(df_blur['category'].value_counts())

# Perform Stratified Split (70/15/15)
train_list, val_list, test_list = [], [], []

for cat in ['Mild', 'Medium', 'Severe']:
    cat_files = df_blur[df_blur['category'] == cat]['filename'].tolist()
    tr, temp = train_test_split(cat_files, test_size=0.3, random_state=42)
    vl, ts = train_test_split(temp, test_size=0.5, random_state=42)
    train_list.extend(tr)
    val_list.extend(vl)
    test_list.extend(ts)

print(f"\nFinal Stratified Splits: Train({len(train_list)}), Val({len(val_list)}), Test({len(test_list)})")

Calculating blur scores...


100%|██████████| 1029/1029 [01:06<00:00, 15.47it/s]



Blur Category Distribution:
category
Mild      350
Severe    340
Medium    339
Name: count, dtype: int64

Final Stratified Splits: Train(720), Val(154), Test(155)


In [30]:
def move_stratified_files(df, split_list, split_name):
    print(f"Moving {split_name} files...")
    for fname in tqdm(split_list):
        # Get category for this specific file
        category = df[df['filename'] == fname]['category'].values[0]

        for folder_type in ["blur", "sharp"]:
            # New path structure: processed / split / category / folder_type
            dest = base_path / "processed_stratified" / split_name / category / folder_type
            dest.mkdir(parents=True, exist_ok=True)

            src_file = base_path / folder_type / "images" / fname
            if src_file.exists():
                shutil.copy(src_file, dest / fname)

# Execute the move for each split
move_stratified_files(df_blur, train_list, 'train')
move_stratified_files(df_blur, val_list, 'val')
move_stratified_files(df_blur, test_list, 'test')

print("\nAll files have been organized into stratified folders.")

Moving train files...


100%|██████████| 720/720 [02:59<00:00,  4.00it/s]


Moving val files...


100%|██████████| 154/154 [00:33<00:00,  4.65it/s]


Moving test files...


100%|██████████| 155/155 [00:34<00:00,  4.45it/s]


All files have been organized into stratified folders.


In [31]:

import time
from skimage import restoration, metrics


# 1. Generate a Motion Blur Point Spread Function (PSF)
def get_motion_psf(size=15, angle=45):
    """Creates a linear motion blur kernel."""
    psf = np.zeros((size, size))
    center = size // 2
    cv2.line(psf, (0, center), (size - 1, center), 1, thickness=1)
    M = cv2.getRotationMatrix2D((center, center), angle, 1.0)
    psf = cv2.warpAffine(psf, M, (size, size))
    return psf / psf.sum()

# 2. Advanced Richardson-Lucy with Edge Handling
def deblur_richardson_lucy_rgb(image, psf, iterations=30):
    """Deblurs an RGB image with edge-padding to prevent ringing artifacts."""
    pad_size = psf.shape[0]
    # Symmetric padding helps mitigate edge effects in FFT-based deconvolution
    padded_img = np.pad(image, ((pad_size, pad_size), (pad_size, pad_size), (0, 0)), mode='symmetric')
    restored = np.zeros_like(padded_img, dtype=np.float64)

    for c in range(3):
        channel = padded_img[:, :, c] / 255.0
        # restoration.richardson_lucy handles the iterative deconvolution
        restored[:, :, c] = restoration.richardson_lucy(channel, psf, num_iter=iterations)

    restored_cropped = restored[pad_size:-pad_size, pad_size:-pad_size, :]
    return np.clip(restored_cropped * 255, 0, 255).astype(np.uint8)

# 3. Batch Processing and Metric Tracking
def process_stratified_test(base_stratified_path, category='Medium', num_samples=3):
    test_dir = Path(base_stratified_path) / "test" / category
    blur_dir = test_dir / "blur"
    sharp_dir = test_dir / "sharp"

    filenames = sorted([f.name for f in blur_dir.glob("*.png")])[:num_samples]
    psf = get_motion_psf(size=15, angle=0)

    results = []
    print(f"Deblurring {num_samples} images from {category} category...")

    for fname in filenames:
        blur_img = cv2.cvtColor(cv2.imread(str(blur_dir / fname)), cv2.COLOR_BGR2RGB)
        sharp_img = cv2.cvtColor(cv2.imread(str(sharp_dir / fname)), cv2.COLOR_BGR2RGB)

        start_time = time.time()
        deblurred_img = deblur_richardson_lucy_rgb(blur_img, psf, iterations=20)
        elapsed = time.time() - start_time

        # Grayscale metrics for standard comparison
        sharp_g = cv2.cvtColor(sharp_img, cv2.COLOR_RGB2GRAY)
        deblur_g = cv2.cvtColor(deblurred_img, cv2.COLOR_RGB2GRAY)

        psnr = metrics.peak_signal_noise_ratio(sharp_g, deblur_g)
        ssim = metrics.structural_similarity(sharp_g, deblur_g, data_range=255)

        results.append({"File": fname, "PSNR": round(psnr, 2), "SSIM": round(ssim, 4), "Time": round(elapsed, 2)})
        print(f"Done: {fname} | PSNR: {psnr:.2f}")

    return pd.DataFrame(results)

# Execute on the stratified processed data
strat_path = base_path / "processed_stratified"
results_df = process_stratified_test(strat_path, category='Medium', num_samples=3)
display(results_df)

Deblurring 3 images from Medium category...
Done: 000020.png | PSNR: 23.70
Done: 000042.png | PSNR: 21.38
Done: 000070.png | PSNR: 20.79


,File,PSNR,SSIM,Time
0,000020.png,23.70,0.7884,9.19
1,000042.png,21.38,0.7577,8.10
2,000070.png,20.79,0.7271,6.34
